# Davies-Harte Circulant Embedding Method

The Cholesky method has a cost of oder $O(n^3) + m \times O(n^2)$ for $m$ sample paths which is extremely inefficient, particularly for a large number of sample paths which is often the case when running monte-carlo simulations.

We implement here the Davies-Harte circulant embedding method which has cost of order $O(m \times nlog(n)$ which is much more efficient. It is however a less theoretically simple than Cholesky as we will see.

The idea is that we expand our covariance matrix $C$ into a circulant matrix $\Pi$ which embeds $C$ (meaning we can easily extract $C$ from $\Pi$). We then notice that $\Pi$ has an LU decomposition where the matrices are equivalent to applying a fourier transform, which allows us to apply a (cost $O(nlogn$)) FFT algorithm to yield a more efficient simulations of the Gaussian distribution (see https://hagerpa.github.io/talks/excersize_sheet_sampling_of_fBm.pdf for more details).

## Generate Multivariate Guassian

In [ ]:
import numpy as np
import scipy as sp

n = 252
T = 365
t = np.linspace(0, T, n+1)

def Davies_Harte(m, t, mu, cov):
    """
    Generate m samples from a multivariate normal distribution with mean mu and covariance cov using the Davies-Harte method.
    Parameters: 
    m (int): Number of samples to generate
    t (array-like): Time points for the series

    mu (array-like): Mean vector of the distribution
    cov (array-like): Covariance matrix of the distribution
    Returns:
    samples (ndarray): An m x n array of samples drawn from the specified multivariate normal distribution
    """
    #step 1: define first row of the circulant matrix
    n = len(t) - 1
    gamma = np.array([cov(0, k) for k in list(range(n + 1)) + list(range(n - 1, -1, -1))])
    
    #step 2: compute the eigenvalues of the circulant matrix & verify non-negativity
    eigenvalues = np.fft.fft(gamma)
    if np.any(eigenvalues < 0):
        raise ValueError("Covariance function does not produce a valid covariance matrix (negative eigenvalues).")
    
    #step 3: repeat the following m times:
    ##3a: generate 2n dimensional vector of i.i.d. standard normal random variables
    Z = np.random.normal(size=(m, 2*n))
    ##3b: compute the product of the square root of the eigenvalues and the Fourier transform of Z
    X = np.fft.fft(np.linalg.diagscaler(np.sqrt(eigenvalues)) @ np.fft.ifft(Z, axis=1)) 
    
    
    
        

    
    

    